# CNN + GRU Training — Driver Drowsiness Detection
**Person 3 — GRU / Temporal Lead**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

## CNN Feature Extractor (from cnn-branch)

In [ ]:
class CNNFeatureExtractor(nn.Module):
    def __init__(self):
        super(CNNFeatureExtractor, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128 * 8 * 8, 128)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

## GRU Temporal Model

In [ ]:
class GRUModel(nn.Module):
    """
    GRU-based temporal model for drowsiness detection.

    Takes CNN feature sequences as input and learns temporal patterns.
    Input shape:  (batch_size, sequence_length, cnn_feature_dim) -> (32, 10, 128)
    Output shape: (batch_size, num_classes) -> (32, 2)
    """
    def __init__(self, input_dim=128, hidden_dim=64, num_layers=2, num_classes=2, dropout=0.3):
        super(GRUModel, self).__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)
        self.batch_norm = nn.BatchNorm1d(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        gru_out, _ = self.gru(x)

        # Use the last time step output
        last_output = gru_out[:, -1, :]

        # Regularization
        out = self.dropout(last_output)
        out = self.batch_norm(out)

        # Classification
        out = self.fc(out)
        return out

## CNN + GRU Combined Model

In [ ]:
class CnnGruModel(nn.Module):
    """
    Combined CNN + GRU model.
    Input:  (batch, seq_len, C, H, W) -> (32, 10, 3, 64, 64)
    Output: (batch, num_classes) -> (32, 2)
    """
    def __init__(self):
        super(CnnGruModel, self).__init__()
        self.cnn = CNNFeatureExtractor()
        self.gru = GRUModel(input_dim=128, hidden_dim=64, num_layers=2, num_classes=2)

    def forward(self, x):
        batch_size, seq_len, C, H, W = x.shape

        # Pass each frame through CNN
        x = x.view(batch_size * seq_len, C, H, W)
        cnn_features = self.cnn(x)

        # Reshape back to sequence
        cnn_features = cnn_features.view(batch_size, seq_len, -1)

        # Pass sequence through GRU
        output = self.gru(cnn_features)
        return output

## Load Data

In [ ]:
data_dir = "/content/drive/MyDrive/processed_data/processed_data"

X_train = np.load(f"{data_dir}/X_train.npy")
y_train = np.load(f"{data_dir}/y_train.npy")
X_val = np.load(f"{data_dir}/X_val.npy")
y_val = np.load(f"{data_dir}/y_val.npy")

# Convert from (N, seq, H, W, C) to (N, seq, C, H, W)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).permute(0, 1, 4, 2, 3)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).permute(0, 1, 4, 2, 3)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

print(f"Train: {X_train_tensor.shape}, Labels: {y_train_tensor.shape}")
print(f"Val:   {X_val_tensor.shape},   Labels: {y_val_tensor.shape}")

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=32, shuffle=False)

## Model Setup

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = CnnGruModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training & Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)
    return total_loss / total, correct / total

## Training Loop

In [ ]:
EPOCHS = 15
train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/content/best_cnn_gru_model.pth")

    print(f"Epoch {epoch}/{EPOCHS} - Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

print(f"\nBest validation accuracy: {best_val_acc:.1%}")

## Results — Loss & Accuracy Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(range(1, EPOCHS+1), train_losses, label='Train Loss', marker='o', color='blue')
ax1.plot(range(1, EPOCHS+1), val_losses, label='Val Loss', marker='s', color='red')
ax1.set_title('CNN+GRU - Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(range(1, EPOCHS+1), train_accs, label='Train Acc', marker='o', color='blue')
ax2.plot(range(1, EPOCHS+1), val_accs, label='Val Acc', marker='s', color='red')
ax2.set_title('CNN+GRU - Accuracy')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('cnn_gru_metrics.png')
plt.show()